In [ ]:
# ---------------- Imports ----------------
import os
import json
import random
import yaml
from collections import defaultdict
import csv



In [ ]:
# ---------------- Args ----------------
DATASET_CHOICE = "20260115T095923-combined-claims-full"

NUMBER_OF_SAMPLES = 40
# Must be even
if NUMBER_OF_SAMPLES % 2 != 0:
    raise ValueError(
        "NUMBER_OF_SAMPLES must be an even number for balanced sampling."
    )

ALLOWED_BIAS_FRAMINGS = {
    "original",
    "authoritative",
    "consensus",
    "prestige",
    "emotional",
    "sensationalist",
}

RANDOM_SEED = 123


In [ ]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
DATA_PATH = os.path.join(PROJ_STORE, "data", "augmented-processed")

DATASET_PATH = os.path.join(DATA_PATH, DATASET_CHOICE)

# OUTPUT
BASE_OUTPUT_DIR = os.path.join(
    PROJ_STORE,
    "evaluation",
    "human-evaluation",
    "raw-batch"
)

FULL_DIR = os.path.join(BASE_OUTPUT_DIR, "full")
BATCH_DIR = os.path.join(BASE_OUTPUT_DIR, "batches")
TEXT_BATCH_DIR = os.path.join(BASE_OUTPUT_DIR, "batches_text")
MERGED_DIR = os.path.join(BASE_OUTPUT_DIR, "batches_merged")
os.makedirs(FULL_DIR, exist_ok=True)
os.makedirs(BATCH_DIR, exist_ok=True)
os.makedirs(TEXT_BATCH_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)


OUTPUT_FILE = os.path.join(
    FULL_DIR,
    f"{DATASET_CHOICE}-{NUMBER_OF_SAMPLES}-batch.csv"
)                     




In [ ]:
# ---------------- Functions ----------------
def load_jsonl_files(dataset_path):

    samples = []

    splits = ["train", "dev", "test"]

    for split in splits:
        split_path = os.path.join(dataset_path, split)

        if not os.path.isdir(split_path):
            continue

        for file in os.listdir(split_path):

            if not file.endswith(".jsonl"):
                continue

            file_path = os.path.join(split_path, file)

            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    samples.append(json.loads(line))

    return samples


def group_by_base_claim_id(samples):

    groups = defaultdict(list)

    for item in samples:
        base_id = item["claim_id"].split(":")[0]
        groups[base_id].append(item)

    return groups


def filter_complete_groups(groups):

    complete_groups = []

    for base_id, items in groups.items():

        framings = {x["framing_type"] for x in items}

        if ALLOWED_BIAS_FRAMINGS.issubset(framings):
            complete_groups.append(items)

    return complete_groups







def split_groups_by_label(groups):

    supports = []
    refutes = []

    for group in groups:

        # Use first item as reference
        label = group[0]["true_label"]

        if label == "SUPPORTS":
            supports.append(group)

        elif label == "REFUTES":
            refutes.append(group)

    return supports, refutes



def validate_complete_groups(groups):

    required = ALLOWED_BIAS_FRAMINGS

    for group in groups:

        framings = {x["framing_type"] for x in group}

        if framings != required:
            raise ValueError(
                f"Incomplete or invalid group: {framings}"
            )

def check_batch_feasibility(groups, batch_size=20):

    G = len(groups)
    total_items = 6 * G

    if G < batch_size:
        raise ValueError(
            f"Need at least {batch_size} groups, got {G}"
        )

    if total_items % batch_size != 0:
        raise ValueError(
            f"Total items {total_items} not divisible by {batch_size}"
        )

    if G % 10 != 0:
        raise ValueError(
            f"Number of groups {G} must be divisible by 10"
        )

    num_batches = total_items // batch_size

    return num_batches


def build_perfect_batches(groups, batch_size=20, seed=42):


    random.seed(seed)

    # Shuffle groups and items inside groups
    shuffled_groups = []

    for group in groups:
        g = group.copy()
        random.shuffle(g)
        shuffled_groups.append(g)

    random.shuffle(shuffled_groups)

    G = len(shuffled_groups)
    num_batches = check_batch_feasibility(
        shuffled_groups,
        batch_size
    )

    # Initialize empty batches
    batches = [[] for _ in range(num_batches)]

    batch_idx = 0

    # Distribute in round-robin
    for i in range(G):

        group = shuffled_groups[i]

        for j in range(len(group)):

            item = group[j]

            placed = False

            # Try to place in next available batch
            for _ in range(num_batches):

                batch = batches[batch_idx]

                used_groups = {
                    x["claim_id"].split(":")[0]
                    for x in batch
                }

                base_id = item["claim_id"].split(":")[0]

                if (
                    len(batch) < batch_size
                    and base_id not in used_groups
                ):
                    batch.append(item)
                    placed = True
                    batch_idx = (batch_idx + 1) % num_batches
                    break

                batch_idx = (batch_idx + 1) % num_batches

            if not placed:
                raise RuntimeError(
                    "Failed to place item. Dataset may be malformed."
                )

    # Final validation
    validate_batches(batches, batch_size)

    return batches



def save_text_only_batches(batches, output_dir, prefix):


    paths = []

    for i, batch in enumerate(batches):

        path = os.path.join(
            output_dir,
            f"{prefix}-batch-{i:03d}.csv"
        )

        with open(path, "w", newline="", encoding="utf-8") as f:

            writer = csv.writer(f)

            # Only one column
            writer.writerow(["Claim"])

            for item in batch:
                writer.writerow([item["restated_claim"]])

        paths.append(path)

    return paths

def validate_batches(batches, batch_size):


    for i, batch in enumerate(batches):

        if len(batch) != batch_size:
            raise ValueError(
                f"Batch {i} size {len(batch)} != {batch_size}"
            )

        base_ids = [
            x["claim_id"].split(":")[0]
            for x in batch
        ]

        if len(base_ids) != len(set(base_ids)):
            raise ValueError(
                f"Duplicate group in batch {i}"
            )
            
        

def save_batches_to_csv(batches, output_dir, prefix):


    paths = []

    for i, batch in enumerate(batches):

        path = os.path.join(
            output_dir,
            f"{prefix}-batch-{i:03d}.csv"
        )

        with open(path, "w", newline="", encoding="utf-8") as f:

            writer = csv.writer(f)

            writer.writerow([
                "base_claim_id",
                "full_claim_id",
                "framing_type",
                "true_label",
                "text"
            ])

            for item in batch:

                base_id = item["claim_id"].split(":")[0]

                writer.writerow([
                    base_id,
                    item["claim_id"],
                    item["framing_type"],
                    item.get("true_label", ""),
                    item["restated_claim"]
                ])

        paths.append(path)

    return paths


def save_full_list(groups, output_file):


    import csv

    with open(output_file, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "base_claim_id",
            "full_claim_id",
            "framing_type",
            "true_label",
            "text"
        ])

        for group in groups:
            for item in group:

                base_id = item["claim_id"].split(":")[0]

                writer.writerow([
                    base_id,
                    item["claim_id"],
                    item["framing_type"],
                    item.get("true_label", ""),
                    item["restated_claim"]
                ])






def save_merged_structured_batches(batches, output_dir, filename):



    path = os.path.join(output_dir, filename)


    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)


        writer.writerow([
            "batch_id",
            "base_claim_id",
            "full_claim_id",
            "framing_type",
            "true_label",
            "text"
        ])


        for batch_idx, batch in enumerate(batches):
            for item in batch:
                base_id = item["claim_id"].split(":")[0]


                writer.writerow([
                    batch_idx,
                    base_id,
                    item["claim_id"],
                    item["framing_type"],
                    item.get("true_label", ""),
                    item["restated_claim"]
                ])


    return path



def save_merged_text_only_batches(batches, output_dir, filename):


    path = os.path.join(output_dir, filename)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Single column only
        writer.writerow(["Claim"])

        for batch in batches:
            for item in batch:
                writer.writerow([
                    item["restated_claim"]
                ])

    return path






    
def main(n, seed=42):

    # Load data
    samples = load_jsonl_files(DATASET_PATH)
    print(f"Loaded {len(samples)} samples")

    # Group by claim id
    grouped = group_by_base_claim_id(samples)
    print(f"Found {len(grouped)} claim groups")

    # Filter complete groups
    complete_groups = filter_complete_groups(grouped)
    print(f"Found {len(complete_groups)} complete groups")

    # Enforce exact structure
    validate_complete_groups(complete_groups)

    # Balanced sampling
    supports, refutes = split_groups_by_label(complete_groups)

    half = n // 2

    if len(supports) < half or len(refutes) < half:
        raise ValueError("Insufficient balanced groups")

    random.seed(seed)

    selected_groups = (
        random.sample(supports, half)
        + random.sample(refutes, half)
    )

    random.shuffle(selected_groups)

    # Check batching math
    num_batches = check_batch_feasibility(selected_groups)

    print(f"Will create {num_batches} batches")

    # Build batches
    batches = build_perfect_batches(
        selected_groups,
        seed=seed
    )

    return selected_groups, batches

In [ ]:
# ---------------- Run ----------------
groups, batches = main(
    NUMBER_OF_SAMPLES,
    seed=RANDOM_SEED
)

print(f"Sampled groups: {len(groups)}")
print(f"Generated batches: {len(batches)}")

# Save full list
save_full_list(groups, OUTPUT_FILE)

print(f"Saved full list: {OUTPUT_FILE}")

# Save full batches
batch_files = save_batches_to_csv(
    batches,
    BATCH_DIR,
    DATASET_CHOICE
)

# Save text-only batches
text_batch_files = save_text_only_batches(
    batches,
    TEXT_BATCH_DIR,
    DATASET_CHOICE
)

print(f"Saved {len(batch_files)} batch files")

print("Example batch:", batch_files[0])



merged_structured = save_merged_structured_batches(
    batches,
    MERGED_DIR,
    f"{DATASET_CHOICE}-{NUMBER_OF_SAMPLES}-all-batches.csv"
)

merged_text_only = save_merged_text_only_batches(
    batches,
    MERGED_DIR,
    f"{DATASET_CHOICE}-{NUMBER_OF_SAMPLES}-all-batches-text.csv"
)

print(f"Saved merged structured batches: {merged_structured}")
print(f"Saved merged text-only batches: {merged_text_only}")








    